# Aggregate Functions and Grouping

So far each query returned individual rows. Often you want a summary instead:
how many books are there, what is the average track length, how many members
come from each country. This notebook covers three related ideas, in order:

1. **Aggregate functions** (COUNT, SUM, AVG, MIN, MAX) that summarize many
   rows into one value.
2. **GROUP BY**, which applies an aggregate separately to each group of rows.
3. **HAVING**, which filters groups after aggregation.

Take them one at a time, each part builds on the one before.

## Part A: Aggregate functions

An aggregate function takes many rows and returns a single value. The five
most common are:

- **COUNT**: how many rows
- **SUM**: total of a numeric column
- **AVG**: average of a numeric column
- **MIN**: smallest value
- **MAX**: largest value

Let's start with COUNT. How many artists are in Chinook?

In [ ]:
from rel_db_learning import get_chinook_connection, get_project_connection, run_sql
from rel_db_learning.widgets.self_assessment import sql_query_check

chinook = get_chinook_connection()
project = get_project_connection()

run_sql("SELECT COUNT(*) AS artist_count FROM artist", chinook)

The `AS artist_count` part gives the result column a friendly name. This is
called an alias.

SUM, AVG, MIN, and MAX work on numeric columns. The track table has a
`milliseconds` column (track length). Let's summarize it:

In [ ]:
run_sql("""
    SELECT
        COUNT(*) AS track_count,
        AVG(milliseconds) AS avg_length,
        MIN(milliseconds) AS shortest,
        MAX(milliseconds) AS longest
    FROM track
""", chinook)

You can combine an aggregate with WHERE to summarize a subset. The average
length of tracks longer than 5 minutes (300000 milliseconds):

In [ ]:
run_sql("SELECT AVG(milliseconds) AS avg_length FROM track WHERE milliseconds > 300000", chinook)

On your Library, COUNT is the most useful aggregate so far. How many books do
you have?

In [ ]:
run_sql("SELECT COUNT(*) AS book_count FROM book", project)

## Part B: GROUP BY

A single COUNT over the whole table is useful, but often you want a count
**per category**. For example, not "how many books in total" but "how many
books per publisher city". That is what GROUP BY does: it splits rows into
groups and applies the aggregate to each group separately.

```sql
SELECT group_column, AGGREGATE(...) FROM table GROUP BY group_column;
```

In Chinook, how many albums does each artist have? (showing a few)

In [ ]:
run_sql("""
    SELECT artist_id, COUNT(*) AS album_count
    FROM album
    GROUP BY artist_id
    ORDER BY album_count DESC
    LIMIT 10
""", chinook)

The rule to remember: every column in the SELECT must either be in the
GROUP BY, or be inside an aggregate function. Here `artist_id` is grouped,
and `COUNT(*)` is an aggregate.

On your Library, how many books come from each publisher city?

In [ ]:
run_sql("""
    SELECT publisher_city, COUNT(*) AS book_count
    FROM book
    GROUP BY publisher_city
    ORDER BY publisher_city
""", project)

## Part C: HAVING

WHERE filters individual rows before grouping. To filter the **groups
themselves** after aggregation, use HAVING. Think of HAVING as "WHERE for
groups".

```sql
SELECT group_column, AGGREGATE(...) FROM table
GROUP BY group_column
HAVING condition_on_aggregate;
```

In Chinook, which artists have more than 5 albums?

In [ ]:
run_sql("""
    SELECT artist_id, COUNT(*) AS album_count
    FROM album
    GROUP BY artist_id
    HAVING COUNT(*) > 5
    ORDER BY album_count DESC
""", chinook)

The difference between WHERE and HAVING matters:
- **WHERE** filters rows before they are grouped.
- **HAVING** filters groups after the aggregate is computed.

On your Library, which publishers have more than 6 books?

In [ ]:
run_sql("""
    SELECT publisher_name, COUNT(*) AS book_count
    FROM book
    GROUP BY publisher_name
    HAVING COUNT(*) > 6
    ORDER BY publisher_name
""", project)

## Exercises

**Exercise 1 (aggregate):** Return the total number of members in your
Library, as a single value. Use COUNT.

In [ ]:
my_sql_1 = "???"

sql_query_check(
    student_sql=my_sql_1,
    correct_sql="m04_nb_05_exercise_1",
    connection=project,
)

**Exercise 2 (GROUP BY):** Return each `publisher_city` together with the
number of books from that city. Use GROUP BY.

In [ ]:
my_sql_2 = "???"

sql_query_check(
    student_sql=my_sql_2,
    correct_sql="m04_nb_05_exercise_2",
    connection=project,
)

**Exercise 3 (GROUP BY):** Return each `country` together with the number of
members from that country.

In [ ]:
my_sql_3 = "???"

sql_query_check(
    student_sql=my_sql_3,
    correct_sql="m04_nb_05_exercise_3",
    connection=project,
)

**Exercise 4 (HAVING):** Return each `publisher_name` and its book count, but
only for publishers with more than 6 books. Use GROUP BY and HAVING.

In [ ]:
my_sql_4 = "???"

sql_query_check(
    student_sql=my_sql_4,
    correct_sql="m04_nb_05_exercise_4",
    connection=project,
)

## The same in Python with Ibis

Ibis expresses aggregation and grouping too. Here is a grouped count on
Chinook: the number of albums per artist.

In [ ]:
import ibis
from rel_db_learning import get_ibis_chinook

ibis_chinook = get_ibis_chinook()
album = ibis_chinook.table("album")

# Albums per artist
result = album.group_by("artist_id").aggregate(album_count=album.count())
result.execute()

And the SQL Ibis generated:

In [ ]:
print(ibis.to_sql(result))

### Optional: try it on your Library

Connect to your Library with Ibis and try a grouped count. The cell below
counts members per country. Try changing the grouping column, for example
group books by publisher_city.

This is optional and not checked.

In [ ]:
from rel_db_learning import get_ibis_project

ibis_project = get_ibis_project()
member = ibis_project.table("member")

member.group_by("country").aggregate(member_count=member.count()).execute()

## Experiment freely

Try other groupings and aggregates. For example, count loans per member, or
count books per genre text. Mix WHERE (filter rows) with HAVING (filter
groups).

In [ ]:
run_sql("""
    SELECT publisher_name, COUNT(*) AS book_count
    FROM book
    GROUP BY publisher_name
    ORDER BY book_count DESC
""", project)

## What's next

You have now learned the core tools for querying a single table: selecting columns, filtering rows, sorting, limiting, removing duplicates, and summarizing with aggregates and grouping. The next notebook is a set of practice exercises that combine everything so far on the Chinook database, before we move on to the next notebook.